In [ ]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

In [ ]:
# Paths -- edit these for your environment.
RAW_DIR  = "/PATH/TO/raw"              # raw MGB extracts (edw/, clarity/, Datamart/)
DATA_DIR = "/PATH/TO/data"             # project data root; lookup tables live here
WORK_DIR = f"{DATA_DIR}/20250604"      # identified intermediate outputs -- keep on the secure server
DEID_DIR = f"{WORK_DIR}/deidentified"  # de-identified outputs and ID / date-shift maps

In [ ]:
base_cohort = pd.read_csv(f'{WORK_DIR}/microbiology_cultures_cohort.csv')
base_cohort

In [ ]:
# TODO:
procedure_df = pd.read_csv(f"{RAW_DIR}/clarity/2025-05-19.done/cpt_2025-05-19.csv")
procedure_df = procedure_df.rename(columns={'patientID':'PatientID',
                                             'patientEncounterID':'PatientEncounterID',
                                             'cptCodeDT':'visit_date'})
procedure_df

In [ ]:
nursing_procedures = procedure_df[((pd.to_numeric(procedure_df['cptCD'], errors='coerce') >= 99304) & (pd.to_numeric(procedure_df['cptCD'], errors='coerce') <= 99318)) | (procedure_df['cptCodeDSC'].str.contains("nursing", case=False, na=False))]
nursing_procedures = nursing_procedures[['PatientID','PatientEncounterID','visit_date']].copy()
nursing_procedures['source'] = 'procedure'
nursing_procedures['nursing_home_visit']='Yes'
nursing_procedures

In [ ]:
admission = pd.read_csv(f'{RAW_DIR}/edw/encounters/MGB_encounters_20240715.csv')
admission

In [ ]:
admission['dischargeDisposition'].unique()

In [ ]:
admission_nursing_home_visits = admission[admission['AdmitSourceDSC'].isin(['Skilled Nursing Facility'])].copy()
admission_nursing_home_visits['visit_date'] = pd.to_datetime(admission_nursing_home_visits['hospitalAdmitDTS']).dt.date
admission_nursing_home_visits = admission_nursing_home_visits[['PatientID','PatientEncounterID',
                                                               'visit_date']].copy()
admission_nursing_home_visits['source'] = 'admission'
admission_nursing_home_visits['nursing_home_visit'] = 'Yes'
admission_nursing_home_visits


In [ ]:
discharge_nursing_home_visits = admission[admission['dischargeDisposition'].isin(['Skilled Nursing Facility','Nursing Facility'])].copy()
discharge_nursing_home_visits['visit_date'] = pd.to_datetime(discharge_nursing_home_visits['hospitalDischargeDTS']).dt.date
discharge_nursing_home_visits = discharge_nursing_home_visits[['PatientID','PatientEncounterID',
                                                               'visit_date']].copy()
discharge_nursing_home_visits['source'] = 'encounter'
discharge_nursing_home_visits['nursing_home_visit'] = 'Yes'
discharge_nursing_home_visits


In [ ]:
admission['dischargeDisposition'].unique()

In [ ]:
nursing_home_visits_temp = pd.concat([nursing_procedures,
                                      admission_nursing_home_visits,
                                      discharge_nursing_home_visits],
                                     ignore_index=True)
nursing_home_visits_temp = nursing_home_visits_temp.rename(columns={'PatientID':'anon_id',
                                                                    'PatientEncounterID':'pat_enc_csn_id_coded'})
nursing_home_visits_temp

In [ ]:
merged = pd.merge(
    base_cohort,
    nursing_home_visits_temp,
    on=['anon_id', 'pat_enc_csn_id_coded'],
    how='left'
)
merged['order_time_jittered_utc'] = pd.to_datetime(merged['order_time_jittered_utc']).dt.date
merged['visit_date'] = pd.to_datetime(merged['visit_date'])

merged['nursing_home_visit_culture'] = (pd.to_datetime(merged['order_time_jittered_utc']) - pd.to_datetime(merged['visit_date'])).dt.days

merged['nursing_home_visit_culture'] = np.where(merged['nursing_home_visit_culture'] < 0, np.nan, merged['nursing_home_visit_culture'])

merged

In [ ]:
# Drop duplicates to mimic GROUP BY in SQL
# Bug fix: variable was incorrectly named 'filtered' — undefined. Should be 'merged'.
final_df = merged[['anon_id','pat_enc_csn_id_coded', 'order_proc_id_coded', 
                     'order_time_jittered_utc', 'visit_date','nursing_home_visit_culture']]

final_df = final_df.drop_duplicates(
    subset=['anon_id', 'pat_enc_csn_id_coded', 'order_proc_id_coded', 'order_time_jittered_utc', 'visit_date']
).sort_values(
    by=['anon_id', 'pat_enc_csn_id_coded', 'order_proc_id_coded', 'order_time_jittered_utc', 'visit_date']
)

final_df

In [ ]:
final_df.to_csv(f'{WORK_DIR}/microbiology_cultures_nursing_home_visits_augmented.csv',
                index = False)

## Deidentification

In [ ]:
# Load the saved mappings
anon_map_df = pd.read_csv(f'{DEID_DIR}/anon_id_map.csv')
enc_map_df = pd.read_csv(f'{DEID_DIR}/enc_id_map.csv')
order_proc_map_df = pd.read_csv(f'{DEID_DIR}/order_proc_id_map.csv')

In [ ]:
# Convert to Dict
anon_id_map = dict(zip(anon_map_df['anon_id'], anon_map_df['anon_id_deid']))
enc_id_map = dict(zip(enc_map_df['pat_enc_csn_id_coded'], enc_map_df['pat_enc_csn_id_deid']))
order_proc_map = dict(zip(order_proc_map_df['order_proc_id_coded'], order_proc_map_df['order_proc_id_deid']))

# Apply the mappings
final_table_deid = final_df.copy()
final_table_deid['anon_id'] = final_table_deid['anon_id'].map(anon_id_map)
final_table_deid['pat_enc_csn_id_coded'] = final_table_deid['pat_enc_csn_id_coded'].map(enc_id_map)
final_table_deid['order_proc_id_coded'] = final_table_deid['order_proc_id_coded'].map(order_proc_map)

final_table_deid

In [ ]:
final_table_deid.to_csv(f'{DEID_DIR}/microbiology_cultures_nursing_home_visits_augmented.csv',
                        index = False)